# 02 — Feature engineering

## Obiettivi didattici

1. Trasformare il timestamp in **feature temporali** atomiche.
2. Calcolare **distanza Haversine** fra cliente e merchant.
3. Costruire **aggregati expanding per cliente** SENZA leakage temporale.
4. Verificare la composizione del DataFrame post-FE.
!!! note "Dataset richiesto"
    Il dataset Kaggle (~470MB) NON e' in repo per limiti di GitHub.
    Scaricalo da <https://www.kaggle.com/datasets/kartik2112/fraud-detection>
    e copia `fraudTrain.csv` e `fraudTest.csv` in `data/raw/`.


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from fraud_pipeline.data import load_train_test, downsample_for_smoke_test
from fraud_pipeline.features import FraudFeatureEngineer
from fraud_pipeline.preprocessing import build_preprocessor, infer_column_groups
from fraud_pipeline.config import DEFAULT_CONFIG


## Carichiamo un sample (50k righe) per velocita'

Il feature engineering completo su 1.5M righe richiede ~30 secondi; su 50k e' istantaneo e didattico.

In [ ]:
df_train, _ = load_train_test()
df_train = downsample_for_smoke_test(df_train, n_rows=50_000,
                                     random_state=DEFAULT_CONFIG.random_state)
print(f'Sample: {df_train.shape}, frodi={df_train.is_fraud.sum()}')
df_train.head(2)


## FraudFeatureEngineer: 3 famiglie di feature

**Temporali** (da `trans_date_trans_time` e `dob`):
- `hour`, `day_of_week`, `month`, `is_weekend`, `is_night`
- `customer_age_years`

**Geografiche** (da `lat/long` cliente e merchant):
- `distance_km` (Haversine)
- `is_far_tx` (>500 km)

**Trasformate dell'importo**:
- `log_amt`, `is_small_amt`

**Aggregati cliente** (expanding, NO leakage):
- `customer_tx_count_so_far`
- `customer_mean_amt_so_far`, `customer_std_amt_so_far`
- `customer_amt_zscore` (deviazione vs storico)


In [ ]:
fe = FraudFeatureEngineer()
X = df_train.drop(columns=['is_fraud'])
X_fe = fe.fit_transform(X)
added = sorted(set(X_fe.columns) - set(X.columns))
print(f'Feature aggiunte ({len(added)}):')
for c in added:
    print(f'  - {c}: dtype={X_fe[c].dtype}')
print(f'\nColonne droppate dopo FE: {sorted(set(X.columns) - set(X_fe.columns))}')


## Verifica no-leakage degli aggregati

Per ogni cliente, la **prima** transazione cronologica deve avere `customer_tx_count_so_far == 0`. Se non fosse cosi', l'expanding starebbe includendo la riga corrente nel suo stesso aggregato.

In [ ]:
verify = X_fe.assign(cc_num=X['cc_num'].values)
first_per_customer = verify.sort_index().groupby('cc_num').head(1)
violations = (first_per_customer['customer_tx_count_so_far'] != 0).sum()
print(f'Violazioni leakage (atteso=0): {violations}')
first_per_customer[['customer_tx_count_so_far',
                    'customer_mean_amt_so_far',
                    'customer_amt_zscore']].head()


## Z-score vs storico: una feature potente

`customer_amt_zscore` misura quanto la transazione corrente devia da quelle storiche dello stesso cliente. Frodi spesso hanno z-score elevato (importo anomalo). Visualizziamo la distribuzione condizionata.

In [ ]:
df_check = X_fe.copy()
df_check['is_fraud'] = df_train['is_fraud'].values
fig, ax = plt.subplots(figsize=(9, 4)) if False else (None, None)
import matplotlib.pyplot as plt; import seaborn as sns
fig, ax = plt.subplots(figsize=(9, 4))
for label, color in [(0, '#4C72B0'), (1, '#C44E52')]:
    sub = df_check.loc[df_check.is_fraud == label, 'customer_amt_zscore'].clip(-5, 15)
    sns.kdeplot(sub, ax=ax, label=f'is_fraud={label}', color=color, lw=2)
ax.set_title('customer_amt_zscore per classe (clipped)')
ax.legend(); plt.show()


## ColumnTransformer (preprocessor)

Due branch parallele:

- **numeriche** -> SimpleImputer(median) + StandardScaler
- **nominali** -> SimpleImputer('unknown') + OneHotEncoder(min_frequency=10)

Le colonne ad altissima cardinalita' (`merchant`, 693 unique) sono droppate per non esplodere la dimensionalita'. In una pipeline avanzata si userebbe target encoding.

In [ ]:
groups = infer_column_groups(X_fe)
print(f'Numeriche: {len(groups["numeric"])}')
print(f'Nominali : {len(groups["nominal"])}  ({groups["nominal"]})')
preproc = build_preprocessor(
    numeric_cols=groups['numeric'],
    nominal_cols=groups['nominal'],
)
preproc


In [ ]:
X_t = preproc.fit_transform(X_fe)
n_in = X_fe.shape[1]
n_out = X_t.shape[1]
print(f'Feature in:  {n_in}')
print(f'Feature out: {n_out}  (espansione +{n_out - n_in} colonne dovuta a OneHot)')


## Conclusione

Il feature engineer e' un **transformer sklearn**: serializzabile, componibile in `Pipeline`, garantisce no-leakage in CV. Pronto per il notebook **03_models_baseline_vs_ensemble**.
